### Integrated Future & Career Prediction Engine

----


| | Model | Algorithm | Task |
|---|---|---|---|
| A | Academic Risk | XGBoost + Random Forest | Classification (Low/Medium/High) |
| B | Career Readiness | XGBoost + Random Forest | Regression (score 0–100) |

---
### Imports files

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor

from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    mean_absolute_error, mean_squared_error, r2_score
)

print('All libraries imported successfully')

---
### Load Preprocessed Data

We load everything that was saved by `dataset_preprocessing.py`.

| File | Contains | Used For |
|---|---|---|
| `risk_train.pkl` | X_train (SMOTE balanced), y_train | Training Model A |
| `risk_test.pkl` | X_test (real data), y_test | Evaluating Model A |
| `career_train.pkl` | X_train (scaled), y_train | Training Model B |
| `career_test.pkl` | X_test (scaled), y_test | Evaluating Model B |
| `feature_columns.pkl` | list of 25 feature names | Labelling importance charts |

In [ ]:
# Adjust this path to match your saved_objects location
SAVED = '../../trained-models/career-prediction-engine/saved_objects/'

X_train_r, y_train_r = joblib.load(SAVED + 'risk_train.pkl')
X_test_r,  y_test_r  = joblib.load(SAVED + 'risk_test.pkl')
X_train_c, y_train_c = joblib.load(SAVED + 'career_train.pkl')
X_test_c,  y_test_c  = joblib.load(SAVED + 'career_test.pkl')
feature_cols         = joblib.load(SAVED + 'feature_columns.pkl')

print(f'Risk   train: {X_train_r.shape}  ← SMOTE balanced (3 equal classes)')
print(f'Risk   test : {X_test_r.shape}   ← Real distribution (never touched by SMOTE)')
print(f'Career train: {X_train_c.shape}')
print(f'Career test : {X_test_c.shape}')
print(f'\nFeatures ({len(feature_cols)}):', feature_cols)

In [ ]:
# Check class distribution in risk test set (should reflect real imbalance)
label_map = {0: 'Low', 1: 'Medium', 2: 'High'}
risk_dist  = pd.Series(y_test_r).map(label_map).value_counts()

print('Risk test set distribution (real):')
print(risk_dist)

fig, ax = plt.subplots(figsize=(5, 3))
risk_dist.plot(kind='bar', ax=ax,
               color=['#2ecc71', '#f39c12', '#e74c3c'],
               edgecolor='white')
ax.set_title('Test Set: Academic Risk Distribution', fontweight='bold')
ax.set_xlabel('Risk Level')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for p in ax.patches:
    ax.annotate(str(int(p.get_height())),
                (p.get_x() + p.get_width() / 2, p.get_height() + 5),
                ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

---
###  Train Model A: Academic Risk Classification

#### Why Two Algorithms?
We train **Random Forest** and **XGBoost** on the same data, then compare them.
The better one becomes the official Model A.

| Algorithm | How It Works (Simple) |
|---|---|
| **Random Forest** | Builds 200 decision trees independently, takes majority vote |
| **XGBoost** | Builds trees sequentially — each tree fixes mistakes of the previous |

In [ ]:
# ── Random Forest Classifier ──
print('Training Random Forest...')
rf_risk = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    class_weight='balanced',   # extra protection on top of SMOTE
    random_state=42,
    n_jobs=-1                  # use all CPU cores
)
rf_risk.fit(X_train_r, y_train_r)
rf_risk_pred = rf_risk.predict(X_test_r)
print('Random Forest trained')

# ── XGBoost Classifier ──
print('Training XGBoost...')
xgb_risk = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)
xgb_risk.fit(X_train_r, y_train_r)
xgb_risk_pred = xgb_risk.predict(X_test_r)
print('XGBoost trained')

---
### Evaluate Model A

### Understanding The Metrics

| Metric | What It Means |
|---|---|
| **Accuracy** | Out of all students, how many did we classify correctly? |
| **Precision** | Of students predicted as High risk, how many really were? |
| **Recall** | Of all actual High risk students, how many did we catch? |
| **F1 Score** | Balance of precision and recall — most important metric |
| **CV Score** | Average accuracy across 5 different train/test splits — measures consistency |

In [ ]:
label_names = ['Low', 'Medium', 'High']

rf_acc  = accuracy_score(y_test_r, rf_risk_pred)
rf_f1   = f1_score(y_test_r, rf_risk_pred, average='weighted')
xgb_acc = accuracy_score(y_test_r, xgb_risk_pred)
xgb_f1  = f1_score(y_test_r, xgb_risk_pred, average='weighted')

print('=' * 50)
print('  RANDOM FOREST — RESULTS')
print('=' * 50)
print(f'  Accuracy  : {rf_acc:.4f}  ({rf_acc*100:.1f}%)')
print(f'  F1 Score  : {rf_f1:.4f}')
print()
print(classification_report(y_test_r, rf_risk_pred, target_names=label_names))

print('=' * 50)
print('  XGBOOST — RESULTS')
print('=' * 50)
print(f'  Accuracy  : {xgb_acc:.4f}  ({xgb_acc*100:.1f}%)')
print(f'  F1 Score  : {xgb_f1:.4f}')
print()
print(classification_report(y_test_r, xgb_risk_pred, target_names=label_names))

In [ ]:
# ── Confusion Matrices Side by Side ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, pred, name, acc in [
    (axes[0], rf_risk_pred,  'Random Forest', rf_acc),
    (axes[1], xgb_risk_pred, 'XGBoost',       xgb_acc)
]:
    cm = confusion_matrix(y_test_r, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_names,
                yticklabels=label_names, ax=ax)
    ax.set_title(f'{name}\nAccuracy: {acc*100:.1f}%', fontsize=13, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.suptitle('Model A — Academic Risk Confusion Matrices', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Reading The Confusion Matrix ──
print('HOW TO READ THE CONFUSION MATRIX:')
print('  Diagonal cells (top-left to bottom-right) = CORRECT predictions')
print('  Off-diagonal cells = mistakes')
print('  Example: Row=High, Col=Medium means:')
print('    student WAS High risk but model predicted Medium — dangerous miss!')

In [ ]:
# ── Cross Validation ── 
# Runs 5 rounds with different train/test splits to check model consistency
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('Running 5-Fold Cross Validation...')
rf_cv  = cross_val_score(rf_risk,  X_train_r, y_train_r, cv=cv, scoring='accuracy')
xgb_cv = cross_val_score(xgb_risk, X_train_r, y_train_r, cv=cv, scoring='accuracy')

print(f'\nRandom Forest CV: {rf_cv.mean():.4f} ± {rf_cv.std():.4f}')
print(f'  Per fold: {[round(s,4) for s in rf_cv]}')
print(f'\nXGBoost CV:       {xgb_cv.mean():.4f} ± {xgb_cv.std():.4f}')
print(f'  Per fold: {[round(s,4) for s in xgb_cv]}')

print()
print('WHAT ± MEANS:')
print('  Small ± (like ±0.002) = model is CONSISTENT across different data splits')
print('  Large ± (like ±0.05)  = model performance varies = less reliable')

In [ ]:
# ── Winner ──
best_risk_model = rf_risk  if rf_f1  >= xgb_f1  else xgb_risk
best_risk_pred  = rf_risk_pred if rf_f1 >= xgb_f1 else xgb_risk_pred
best_risk_name  = 'Random Forest' if rf_f1 >= xgb_f1 else 'XGBoost'
best_risk_acc   = rf_acc if rf_f1 >= xgb_f1 else xgb_acc
best_risk_f1    = rf_f1  if rf_f1 >= xgb_f1 else xgb_f1

print(f'🏆 Model A Winner: {best_risk_name}')
print(f'   Accuracy : {best_risk_acc*100:.2f}%')
print(f'   F1 Score : {best_risk_f1:.4f}')

---
### Train Model B: Career Readiness Regression

### Classification vs Regression — Key Difference

| | Model A (Classification) | Model B (Regression) |
|---|---|---|
| **Output** | A category: Low / Medium / High | A number: e.g. 72.5 |
| **Metric** | Accuracy, F1 Score | MAE, RMSE, R² |
| **SMOTE** | Yes (classes were imbalanced) | No (numbers don't have classes) |

### Understanding Regression Metrics
| Metric | Meaning | Good Value |
|---|---|---|
| **MAE** | Average error in score points. MAE=2 means predictions off by 2 points on average | < 5.0 |
| **RMSE** | Like MAE but punishes large errors more | < 7.0 |
| **R²** | % of variance explained. R²=0.98 means model explains 98% of score variation | > 0.85 |

In [ ]:
# ── Random Forest Regressor ──
print('Training Random Forest Regressor...')
rf_career = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)
rf_career.fit(X_train_c, y_train_c)
rf_career_pred = rf_career.predict(X_test_c)
print('  ✅ Random Forest trained')

# ── XGBoost Regressor ──
print('Training XGBoost Regressor...')
xgb_career = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_career.fit(X_train_c, y_train_c)
xgb_career_pred = xgb_career.predict(X_test_c)
print('  ✅ XGBoost trained')

---
### Evaluate Model B

In [ ]:
rf_mae  = mean_absolute_error(y_test_c, rf_career_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test_c, rf_career_pred))
rf_r2   = r2_score(y_test_c, rf_career_pred)

xgb_mae  = mean_absolute_error(y_test_c, xgb_career_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test_c, xgb_career_pred))
xgb_r2   = r2_score(y_test_c, xgb_career_pred)

results_df = pd.DataFrame({
    'Algorithm'   : ['Random Forest', 'XGBoost'],
    'MAE ↓'       : [rf_mae,  xgb_mae],
    'RMSE ↓'      : [rf_rmse, xgb_rmse],
    'R² Score ↑'  : [rf_r2,   xgb_r2]
})
results_df = results_df.set_index('Algorithm').round(4)
print('Model B — Regression Results:')
print(results_df)
print('\n  ↓ = lower is better   ↑ = higher is better')

In [ ]:
# ── Actual vs Predicted Plots ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, pred, name, r2, mae in [
    (axes[0], rf_career_pred,  'Random Forest', rf_r2,  rf_mae),
    (axes[1], xgb_career_pred, 'XGBoost',       xgb_r2, xgb_mae)
]:
    ax.scatter(y_test_c, pred, alpha=0.3, color='steelblue', s=10)
    mn, mx = y_test_c.min(), y_test_c.max()
    ax.plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect prediction')
    ax.set_title(f'{name}\nR²={r2:.4f}  MAE={mae:.3f}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Actual Career Readiness Score')
    ax.set_ylabel('Predicted Career Readiness Score')
    ax.legend()

plt.suptitle('Model B — Career Readiness: Actual vs Predicted', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('HOW TO READ THIS CHART:')
print('  Red dashed line = perfect prediction (actual = predicted)')
print('  Blue dots close to red line = good predictions')
print('  Dots far from line = errors')
print('  Tight cluster along red line = model is very accurate')

In [ ]:
# ── Error Distribution ──
best_career_pred = xgb_career_pred if xgb_r2 >= rf_r2 else rf_career_pred
best_career_name = 'XGBoost' if xgb_r2 >= rf_r2 else 'Random Forest'

errors = y_test_c.values - best_career_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram of errors
axes[0].hist(errors, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='red', linestyle='--', lw=2, label='Zero error')
axes[0].set_title(f'Prediction Error Distribution\n{best_career_name}', fontweight='bold')
axes[0].set_xlabel('Error (Actual − Predicted)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Residual plot
axes[1].scatter(best_career_pred, errors, alpha=0.3, color='orange', s=10)
axes[1].axhline(0, color='red', linestyle='--', lw=2)
axes[1].set_title(f'Residuals vs Predicted\n{best_career_name}', fontweight='bold')
axes[1].set_xlabel('Predicted Score')
axes[1].set_ylabel('Residual (Actual − Predicted)')

plt.tight_layout()
plt.show()

print('HOW TO READ ERROR DISTRIBUTION:')
print('  Centered at 0 = model is not biased')
print('  Narrow bell shape = most predictions are very close to actual')
print('  Symmetric = model does not over/under predict consistently')

---
### Feature Importance

**What is feature importance?**
It shows which input features the model relied on the most when making predictions.
Higher importance = that feature had more influence on the prediction.

In [ ]:
best_risk_model   = xgb_risk   if xgb_f1  >= rf_f1  else rf_risk
best_career_model = xgb_career if xgb_r2  >= rf_r2  else rf_career

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

domain_colors = {
    'gpa_cumulative': '#3498db', 'gpa_trend': '#3498db',
    'module_avg_score': '#3498db', 'module_score_variance': '#3498db',
    'project_performance': '#3498db', 'assignment_completion_rate': '#3498db',
    'late_submission_rate': '#3498db', 'resit_count': '#3498db',
    'lms_login_frequency': '#3498db',
    'weekly_study_hours': '#2ecc71', 'attendance_rate': '#2ecc71',
    'sleep_hours_avg': '#2ecc71', 'sleep_consistency': '#2ecc71',
    'extracurricular_hours': '#2ecc71', 'part_time_work_hours': '#2ecc71',
    'library_resource_usage': '#2ecc71', 'peer_collaboration_score': '#2ecc71',
    'help_seeking_behavior': '#2ecc71',
    'stress_level': '#e74c3c', 'anxiety_score': '#e74c3c',
    'mood_stability': '#e74c3c', 'motivation_score': '#e74c3c',
    'social_support_score': '#e74c3c', 'sense_of_belonging': '#e74c3c',
    'career_clarity_score': '#9b59b6'
}

for ax, model, name, title in [
    (axes[0], best_risk_model,   best_risk_name,   'Academic Risk Classification'),
    (axes[1], best_career_model, best_career_name, 'Career Readiness Regression')
]:
    imp = pd.Series(model.feature_importances_, index=feature_cols)
    imp = imp.sort_values(ascending=True).tail(15)
    colors = [domain_colors.get(f, '#95a5a6') for f in imp.index]
    imp.plot(kind='barh', ax=ax, color=colors)
    ax.set_title(f'{title}\n{name} — Top 15 Features', fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature Importance Score')
    ax.grid(axis='x', alpha=0.3)

# Legend for domain colors
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3498db', label='Academic'),
    Patch(facecolor='#2ecc71', label='Behavioral'),
    Patch(facecolor='#e74c3c', label='Emotional'),
    Patch(facecolor='#9b59b6', label='Career'),
]
fig.legend(handles=legend_elements, loc='lower center',
           ncol=4, fontsize=11, bbox_to_anchor=(0.5, -0.05))

plt.suptitle('Feature Importance by Domain', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print('COLOR KEY:')
print('  🔵 Blue   = Academic features')
print('  🟢 Green  = Behavioral features')
print('  🔴 Red    = Emotional features')
print('  🟣 Purple = Career features')

In [ ]:
# ── Domain Contribution Pie Charts ──
academic_feats   = ['gpa_cumulative','gpa_trend','module_avg_score','module_score_variance',
                    'project_performance','assignment_completion_rate','late_submission_rate',
                    'resit_count','lms_login_frequency']
behavioral_feats = ['weekly_study_hours','attendance_rate','sleep_hours_avg',
                    'sleep_consistency','extracurricular_hours','part_time_work_hours',
                    'library_resource_usage','peer_collaboration_score','help_seeking_behavior']
emotional_feats  = ['stress_level','anxiety_score','mood_stability','motivation_score',
                    'social_support_score','sense_of_belonging']
career_feats     = ['career_clarity_score']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, model, name, title in [
    (axes[0], best_risk_model,   best_risk_name,   'Academic Risk'),
    (axes[1], best_career_model, best_career_name, 'Career Readiness')
]:
    imp = pd.Series(model.feature_importances_, index=feature_cols)
    domain_sums = {
        'Academic'  : imp[academic_feats].sum(),
        'Behavioral': imp[behavioral_feats].sum(),
        'Emotional' : imp[emotional_feats].sum(),
        'Career'    : imp[career_feats].sum()
    }
    ax.pie(
        list(domain_sums.values()),
        labels=list(domain_sums.keys()),
        colors=['#3498db','#2ecc71','#e74c3c','#9b59b6'],
        autopct='%1.1f%%',
        startangle=90
    )
    ax.set_title(f'{title} ({name})\nDomain Contribution', fontweight='bold')

plt.suptitle('Which Data Domain Matters Most?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
### RF vs XGBoost Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Model A comparison
ax = axes[0]
metrics_a = ['Accuracy', 'F1 Score']
rf_vals_a  = [rf_acc, rf_f1]
xgb_vals_a = [xgb_acc, xgb_f1]
x = np.arange(len(metrics_a))
bars1 = ax.bar(x - 0.2, rf_vals_a,  0.35, label='Random Forest', color='#3498db', alpha=0.85)
bars2 = ax.bar(x + 0.2, xgb_vals_a, 0.35, label='XGBoost',       color='#e74c3c', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(metrics_a)
ax.set_ylim(0.9, 1.01)
ax.set_title('Model A — Academic Risk\nRF vs XGBoost', fontweight='bold')
ax.set_ylabel('Score')
ax.legend()
for bar in bars1: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                           f'{bar.get_height():.3f}', ha='center', fontsize=9)
for bar in bars2: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                           f'{bar.get_height():.3f}', ha='center', fontsize=9)

# Model B comparison
ax = axes[1]
metrics_b  = ['MAE (lower=better)', 'R² Score (higher=better)']
rf_vals_b  = [rf_mae,  rf_r2]
xgb_vals_b = [xgb_mae, xgb_r2]
x = np.arange(len(metrics_b))
bars3 = ax.bar(x - 0.2, rf_vals_b,  0.35, label='Random Forest', color='#3498db', alpha=0.85)
bars4 = ax.bar(x + 0.2, xgb_vals_b, 0.35, label='XGBoost',       color='#e74c3c', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(metrics_b, fontsize=9)
ax.set_title('Model B — Career Readiness\nRF vs XGBoost', fontweight='bold')
ax.set_ylabel('Value')
ax.legend()
for bar in bars3: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                           f'{bar.get_height():.3f}', ha='center', fontsize=9)
for bar in bars4: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                           f'{bar.get_height():.3f}', ha='center', fontsize=9)

plt.suptitle('Algorithm Comparison: Random Forest vs XGBoost', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
### Final Summary

In [ ]:
best_risk_name   = 'XGBoost' if xgb_f1  >= rf_f1  else 'Random Forest'
best_career_name = 'XGBoost' if xgb_r2  >= rf_r2  else 'Random Forest'

print('=' * 62)
print('  TRAINING COMPLETE — FINAL RESULTS SUMMARY')
print('=' * 62)
print()
print('  MODEL A — Academic Risk Classification')
print('  ─────────────────────────────────────')
print(f'  Random Forest → Accuracy: {rf_acc:.4f}  F1: {rf_f1:.4f}')
print(f'  XGBoost       → Accuracy: {xgb_acc:.4f}  F1: {xgb_f1:.4f}')
print(f'  🏆 Winner: {best_risk_name}')
print()
print('  MODEL B — Career Readiness Regression')
print('  ─────────────────────────────────────')
print(f'  Random Forest → MAE: {rf_mae:.4f}  RMSE: {rf_rmse:.4f}  R²: {rf_r2:.4f}')
print(f'  XGBoost       → MAE: {xgb_mae:.4f}  RMSE: {xgb_rmse:.4f}  R²: {xgb_r2:.4f}')
print(f'  🏆 Winner: {best_career_name}')
print()
print('  NOTE ON HIGH ACCURACY:')
print('  These scores are high because the dataset is synthetic.')
print('  Synthetic data has clean mathematical patterns that')
print('  models learn perfectly. This is expected and acceptable')
print('  for a research proof-of-concept system.')
print()
print('  NEXT STEP: What-If Simulation Engine')
print('  Load saved models → accept student input → predict')
print('=' * 62)